# 01. Энкодеры без дообучения

Сравниваю три готовых энкодера (rubert-tiny-turbo, multilingual-e5-base, USER-bge-m3) на своей валидации, чтобы выбрать, какой дообучать. Заодно считаю эмбеддинги e5-base без дообучения: пайплайн использует их как отдельный источник кандидатов.

- Входы, в `MyDrive/avito_dataset/`: `train.parquet`, `benchmark_items.parquet`, `benchmark_queries.parquet` и из `artifacts/` файлы `split.parquet`, `split_queries.parquet`, `corpora.parquet`.
- Выходы: эмбеддинги в `MyDrive/avito_dataset/out/dense_zs/<модель>/` и архив `e5-base-zs.zip`, его нужно распаковать в `artifacts/dense/`. Готовые файлы при перезапуске не пересчитываются.
- Время на T4: около часа на все модели.

Сокращения, они же стоят в именах файлов:
- текст объявления: `t` - только заголовок, `tp` - заголовок и параметры (вид, тип, названия услуг), `tpd` - ещё и начало описания;
- запрос: `q` - только текст, `qf` - текст вместе с фильтрами;
- поиск: `glob` - по всему корпусу, `loc` - только в локальной зоне запроса, `lf` - сначала локальная зона, потом остальное (так ищет пайплайн).

In [ ]:
# версии, с которыми считались эмбеддинги
!pip -q install sentence-transformers==5.2.2 transformers==5.1.0
import os, sys, torch
print(sys.version, "torch", torch.__version__)
assert torch.cuda.is_available(), "Нужен GPU: Runtime -> Change runtime type -> T4 GPU"
!nvidia-smi --query-gpu=name,memory.total --format=csv

In [ ]:
from google.colab import drive
drive.mount("/content/drive")
DRIVE = "/content/drive/MyDrive/avito_dataset"
OUT = f"{DRIVE}/out/dense_zs"
os.makedirs(OUT, exist_ok=True)
print(os.listdir(DRIVE))

## Тексты запросов и объявлений
Та же логика, что `clean`, `item_text` и `query_text` в `src/text.py`.

In [ ]:
# Тексты для e5, та же логика, что clean, item_text и query_text в src/text.py.
import re

DESC_WORDS = 120

_LAT2CYR = str.maketrans("aeopcxykmthb", "аеорсхукмтнв")
_CYR2LAT = str.maketrans("аеорсхукмтнв", "aeopcxykmthb")
_CYR = re.compile(r"[а-я]")
_LAT = re.compile(r"[a-z]")
_LAT_ONLY = re.compile(r"[dfgijlnqrsuvwz]")
_CYR_ONLY = re.compile(r"[бгдежзийлпфцчшщъыьэюя]")
_WORD = re.compile(r"\w+")
_EMOJI = re.compile("[\U0001F000-\U0001FAFF☀-➿⬀-⯿️‍⃣]+")
_REPEAT_PUNCT = re.compile(r"([^\w\s])\1+")
_SPACES = re.compile(r"\s+")


def _fix_word(m):
    w = m.group(0)
    if not (_CYR.search(w) and _LAT.search(w)):
        return w
    lat_only, cyr_only = bool(_LAT_ONLY.search(w)), bool(_CYR_ONLY.search(w))
    if cyr_only and not lat_only:
        return w.translate(_LAT2CYR)
    if lat_only and not cyr_only:
        return w.translate(_CYR2LAT)
    return w.translate(_LAT2CYR) if len(_CYR.findall(w)) >= len(_LAT.findall(w)) else w.translate(_CYR2LAT)


def clean(s: str) -> str:
    s = (s or "").lower().replace("ё", "е")
    if _CYR.search(s) and _LAT.search(s):
        s = _WORD.sub(_fix_word, s)
    s = _EMOJI.sub(" ", s)
    s = _REPEAT_PUNCT.sub(r"\1", s)
    return _SPACES.sub(" ", s).strip()


_VID = re.compile(r"Вид услуги (.+?)(?= Место оказания услуг| Тип услуги| Тип стоимости| Вид услуги|$)")
_TIP = re.compile(r"Тип услуги (?:автосервиса )?(.+?)(?= Место оказания услуг| Вид услуги| Тип услуги| Тип стоимости|$)")
_NAMES = re.compile(r"(?:Название услуги|Услуги) (.+?)(?= Начальная цена| Тип стоимости| Стоимость| Продолжительность|$)")


# Значение обрезаем на первом слове с заглавной буквы: с него начинается следующий ключ.
def _cut(v: str, max_words: int = 8) -> str:
    words = v.split()
    out = words[:1]
    for w in words[1:max_words]:
        if w[:1].isupper():
            break
        out.append(w)
    return " ".join(out)


# Из параметров только вид, тип и до восьми названий услуг.
def params_short(p: str, max_names: int = 8) -> str:
    p = p or ""
    vid = [_cut(v) for v in _VID.findall(p) if v and not v.startswith(("Место", "Тип"))]
    tip = [_cut(v) for v in _TIP.findall(p) if v and not v.startswith(("Место", "Вид"))]
    names = []
    for n in _NAMES.findall(p):
        n = _cut(n.strip())
        if n and n not in names and len(n) < 80:
            names.append(n)
        if len(names) >= max_names:
            break
    parts = []
    if vid:
        parts.append("вид услуги: " + ", ".join(dict.fromkeys(vid)))
    if tip:
        parts.append("тип услуги: " + ", ".join(dict.fromkeys(tip)))
    if names:
        parts.append("услуги: " + "; ".join(names))
    return clean(". ".join(parts))


# rep: t - заголовок, tp - ещё параметры, tpd - ещё начало описания (отсюда items_tpd.npy).
def item_text(title: str, params: str, desc: str, rep: str = "tpd") -> str:
    parts = [clean(title)]
    if rep in ("tp", "tpd"):
        ps = params_short(params)
        if ps:
            parts.append(ps)
    if rep == "tpd":
        d = " ".join(clean(desc).split()[:DESC_WORDS])
        if d:
            parts.append(d)
    return ". ".join(parts)


# с with_filters к запросу дописываются фильтры: q_<часть>_qf.npy, без них q_<часть>_q.npy
def query_text(query: str, filters: str = "", with_filters: bool = False) -> str:
    q = clean(query)
    if with_filters and filters:
        q = q + ". " + clean(filters)
    return q

In [ ]:
import numpy as np, pandas as pd, pyarrow.parquet as pq, time, gc
SEED = 42
MODELS = {
    "rubert-tiny-turbo": dict(hf="sergeyzh/rubert-tiny-turbo", qp="", pp="", reps=["t", "tp", "tpd"], bs=256),
    "e5-base": dict(hf="intfloat/multilingual-e5-base", qp="query: ", pp="passage: ", reps=["t", "tp", "tpd"], bs=128),
    "user-bge-m3": dict(hf="deepvk/USER-bge-m3", qp="", pp="", reps=["tpd"], bs=64),
}
ITEM_MAX_LEN, QUERY_MAX_LEN = 256, 64
KS = [50, 300, 1000]
ALL_RUSSIA = 621540

## Данные

In [ ]:
# Объявления: корпус бенчмарка и ответы val и rank, которых в нём нет.
# Их порядок - порядок строк в items_*.npy, он сохраняется в items_order.parquet.
t0 = time.time()
corp = pd.read_parquet(f"{DRIVE}/corpora.parquet")
need = corp[corp.in_bench | corp.in_val | corp.in_rank]
ICOLS = ["item_id", "item_title_raw", "item_infm_params_text", "item_description_raw", "item_location_id", "item_category_id"]
bi = pd.read_parquet(f"{DRIVE}/benchmark_items.parquet", columns=ICOLS)
extra_ids = sorted(set(need.item_id) - set(bi.item_id))
tr_items = pq.read_table(f"{DRIVE}/train.parquet", columns=ICOLS, filters=[("item_id", "in", extra_ids)]).to_pandas().drop_duplicates("item_id")
items = pd.concat([bi, tr_items], ignore_index=True).drop_duplicates("item_id").reset_index(drop=True)
items = items.merge(need, on="item_id", how="left")
for c in ["in_bench", "in_val", "in_rank"]:
    items[c] = items[c].fillna(False).astype(bool)
del bi, tr_items; gc.collect()

sq = pd.read_parquet(f"{DRIVE}/split_queries.parquet")
queries = {"val": sq[sq.part == "val"].reset_index(drop=True), "rank": sq[sq.part == "rank"].reset_index(drop=True),
           "bench": pd.read_parquet(f"{DRIVE}/benchmark_queries.parquet")}
ITEM_IDX = {iid: i for i, iid in enumerate(items.item_id)}
print(f"items {len(items):,}, queries " + ", ".join(f"{k} {len(v)}" for k, v in queries.items()), f"{time.time()-t0:.0f}s")

In [ ]:
# Локальные зоны запросов, как в src/split.py: регион - локации, дающие 90% выборов по base.
pairs = pd.read_parquet(f"{DRIVE}/train.parquet", columns=["search_location_id", "item_id", "item_location_id"])
split = pd.read_parquet(f"{DRIVE}/split.parquet", columns=["row_id", "part"]).sort_values("row_id")
base = pairs[split.part.values == "base"]
item_locs = set(pairs.item_location_id.unique()) | set(items.item_location_id.unique())
region_locs = set(base.search_location_id.unique()) - item_locs
region_map = {}
for L in region_locs:
    if L == ALL_RUSSIA:
        region_map[L] = None
        continue
    s = base.loc[base.search_location_id == L, "item_location_id"].value_counts(normalize=True)
    region_map[L] = np.array(s.index[: int(np.searchsorted(s.cumsum().values, 0.9)) + 1])
del pairs, split, base; gc.collect()
print("регионов:", len(region_map))

## Кодирование и поиск

In [ ]:
from sentence_transformers import SentenceTransformer

def item_texts(rep):
    return [item_text(t, p, d, rep) for t, p, d in zip(items.item_title_raw, items.item_infm_params_text, items.item_description_raw)]

def query_texts(df, variant):
    return [query_text(q, f, variant == "qf") for q, f in zip(df.search_query, df.search_infm_params_text)]

def load_model(hf):
    return SentenceTransformer(hf, device="cuda").half()

# готовый .npy на Диске не пересчитываем, так переживаем обрыв сессии
def encode_cached(path, model, texts, prefix, max_len, bs):
    if os.path.exists(path):
        return np.load(path)
    model.max_seq_length = max_len
    t0 = time.time()
    emb = model.encode([prefix + t for t in texts], batch_size=bs, normalize_embeddings=True,
                       convert_to_numpy=True, show_progress_bar=True).astype(np.float16)
    np.save(path, emb)
    print(f"  {os.path.basename(path)}: {emb.shape}, {time.time()-t0:.0f}s")
    return emb

In [ ]:
# Маска локальной зоны (запросы x объявления): город - та же локация, регион - его локации, вся Россия - всё.
def local_mask(qloc, loc_t):
    rows = []
    for L in qloc:
        if L in region_map:
            area = region_map[L]
            rows.append(torch.ones_like(loc_t, dtype=torch.bool) if area is None else torch.isin(loc_t, torch.as_tensor(area, device=loc_t.device)))
        else:
            rows.append(loc_t == int(L))
    return torch.stack(rows)

# Топ-K в режимах glob, loc и lf; индексы в порядке items, -1 - пусто.
def search(q_emb, qdf, item_emb, corpus_mask, K=1000, bs=256):
    cidx = np.flatnonzero(corpus_mask)
    I = torch.from_numpy(item_emb[cidx]).to("cuda").half()
    loc_t = torch.from_numpy(items.item_location_id.values[cidx].astype(np.int64)).to("cuda")
    not114 = torch.from_numpy(items.item_category_id.values[cidx] != 114).to("cuda")
    K = min(K, len(cidx))
    out = {m: np.full((len(qdf), K), -1, dtype=np.int64) for m in ["glob", "loc", "lf"]}
    for s in range(0, len(qdf), bs):
        sub = qdf.iloc[s:s + bs]
        Q = torch.from_numpy(q_emb[s:s + bs]).to("cuda").to(I.dtype)
        S = (Q @ I.T).float()
        # фильтр категории: запросам категории 114 только её объявления
        cat114 = torch.from_numpy(sub.search_category.values == 114).to("cuda")
        S[cat114[:, None] & not114[None, :]] = -1e4
        L = local_mask(sub.search_location_id.values, loc_t)
        for mode, M in [("glob", S), ("lf", S + 10.0 * L), ("loc", torch.where(L, S, torch.full_like(S, -1e4)))]:
            v, ix = M.topk(K, dim=1)
            ix = cidx[ix.cpu().numpy()]
            ix[(v < -1e3).cpu().numpy()] = -1
            out[mode][s:s + len(sub)] = ix
    return out

def recall(lists, gold, k):
    return float(np.mean([len(set(l[:k][l[:k] >= 0]) & g) / len(g) for l, g in zip(lists, gold)]))

val = queries["val"]
gold = [set(ITEM_IDX[i] for i in a if i in ITEM_IDX) for a in val.answers]
# recall во всех трёх режимах
def metric_row(res, **info):
    row = dict(info)
    for mode, lists in res.items():
        for k in KS:
            row[f"{mode}@{k}"] = round(recall(lists, gold, k), 4)
    return row

In [ ]:
metrics_path = f"{OUT}/metrics.csv"
rows = pd.read_csv(metrics_path).to_dict("records") if os.path.exists(metrics_path) else []
for mkey, cfg in MODELS.items():
    os.makedirs(f"{OUT}/{mkey}", exist_ok=True)
    items[["item_id"]].to_parquet(f"{OUT}/{mkey}/items_order.parquet", index=False)
    model = load_model(cfg["hf"])
    qemb = {}
    for part, df in queries.items():
        for variant in ["q", "qf"]:
            qemb[(part, variant)] = encode_cached(f"{OUT}/{mkey}/q_{part}_{variant}.npy", model, query_texts(df, variant), cfg["qp"], QUERY_MAX_LEN, 512)
    for rep in cfg["reps"]:
        t0 = time.time()
        iemb = encode_cached(f"{OUT}/{mkey}/items_{rep}.npy", model, item_texts(rep), cfg["pp"], ITEM_MAX_LEN, cfg["bs"])
        t_enc = time.time() - t0
        for variant in ["q", "qf"]:
            if any(r.get("model") == mkey and r.get("rep") == rep and r.get("query") == variant for r in rows):
                continue
            res = search(qemb[("val", variant)], val, iemb, items.in_val.values)
            rows.append(metric_row(res, model=mkey, rep=rep, query=variant, t_encode_items_s=round(t_enc)))
            # сохраняем после каждой конфигурации на случай обрыва сессии
            pd.DataFrame(rows).to_csv(metrics_path, index=False)
            print(rows[-1])
    del model; gc.collect(); torch.cuda.empty_cache()

## Итог

Запуск на T4, val, числа из `metrics.csv`, который пишет этот ноутбук (`MyDrive/avito_dataset/out/dense_zs/metrics.csv`). Прирост к BM25@1000 - насколько вырастает recall, если к топ-1000 BM25 добавить топ-1000 энкодера (lf). Он считался в том же запуске по топ-1000 BM25 для val из пайплайна.

| модель | объявление | запрос | lf@50 | lf@1000 | прирост к BM25@1000 |
|---|---|---|---|---|---|
| rubert-tiny-turbo | t | q | 0.6751 | 0.891 | 0.0454 |
| rubert-tiny-turbo | t | qf | 0.6558 | 0.8859 | 0.045 |
| rubert-tiny-turbo | tp | q | 0.6601 | 0.8871 | 0.045 |
| rubert-tiny-turbo | tp | qf | 0.6997 | 0.9016 | 0.0473 |
| rubert-tiny-turbo | tpd | q | 0.7377 | 0.9187 | 0.0494 |
| rubert-tiny-turbo | tpd | qf | 0.7238 | 0.9152 | 0.0511 |
| e5-base | t | q | 0.6982 | 0.9035 | 0.0449 |
| e5-base | t | qf | 0.6919 | 0.9 | 0.0446 |
| e5-base | tp | q | 0.7481 | 0.9192 | 0.0478 |
| e5-base | tp | qf | 0.7685 | 0.9285 | 0.0513 |
| e5-base | tpd | q | 0.8028 | 0.935 | 0.048 |
| e5-base | tpd | qf | 0.8028 | 0.9374 | 0.0509 |
| user-bge-m3 | tpd | q | 0.8064 | 0.9407 | 0.0514 |
| user-bge-m3 | tpd | qf | 0.7954 | 0.9387 | 0.0529 |

Полный текст объявления (`tpd`) важнее выбора модели: у e5-base lf@50 растёт с 0.6982 (`t`) до 0.8028 (`tpd`). e5-base почти догоняет USER-bge-m3 (0.8028 против 0.8064 по lf@50), а корпус кодирует за 853 секунды против 2047, поэтому дообучал я её. С фильтрами в запросе lf@50 тот же, а прирост к BM25 чуть больше (0.0509 против 0.048), поэтому дальше запрос с фильтрами.

In [ ]:
df = pd.read_csv(metrics_path)
pd.set_option("display.width", 250)
cols = ["model", "rep", "query", "glob@50", "loc@50", "lf@50", "lf@300", "lf@1000", "glob@1000", "t_encode_items_s"]
print(df[cols].sort_values("lf@50", ascending=False).to_string(index=False))

## Скачать результаты

In [ ]:
# Скачиваем архив прямо из сессии: если на Диске кончится место, файлы на него не выгрузятся.
import zipfile
from google.colab import files
ZIP = "/content/e5-base-zs.zip"
with zipfile.ZipFile(ZIP, "w", zipfile.ZIP_STORED) as z:
    for f in ["items_tpd.npy", "items_order.parquet", "q_val_qf.npy", "q_rank_qf.npy", "q_bench_qf.npy"]:
        z.write(f"{OUT}/e5-base/{f}", f"e5-base-zs/{f}")
files.download(ZIP)